# EdgeSHAPer End-to-End Test

This notebook runs an end-to-end pipeline with the project utilities:
- Load SMILES and split data
- Train a real GNN model with `train_gnn_model`
- Save and reload the model with `load_gnn_model`
- Build a graph sample and explain the prediction with `Edgeshaper`
- Compute fidelity and trustworthiness metrics
- Optionally render molecular heatmap visualization

In [1]:
from pathlib import Path
import sys
import csv

import torch

# Ensure src/ is importable when running from notebooks/
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chemagent.ml.gnn_models import GCN
from chemagent.ml.gnn_training import (
    train_gnn_model,
    load_gnn_model,
    smiles_to_nx_graph,
    nx_graph_to_pyg_data,
)
from chemagent.explainability.edgeshaper import Edgeshaper

c:\Repositories\AI-Agent-for-Compound-Prediction-and-Explainability\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import joblib

dataset_csv = repo_root / 'data' / 'datasets' / 'chembl_activity_data_O00329_P42336.csv'
split_path = repo_root / 'splits' / 'chembl_activity_data_O00329_P42336.pkl'
model_dir = repo_root / 'session' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_save_path = model_dir / 'gnn_edgeshaper_test.pt'
hidden_channels = 64

smiles_list = []
with open(dataset_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        smiles_list.append(row['smiles'])

split_obj = joblib.load(split_path)
num_classes = len(set(list(split_obj['train_labels']) + list(split_obj['test_labels'])))

# Device selection: prefer CUDA when available
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_str)

print('Dataset CSV:', dataset_csv)
print('Split file :', split_path)
print('SMILES count:', len(smiles_list))
print('Detected classes:', num_classes)
print('Using device:', device_str)

train_result = train_gnn_model(
    split_file_path=str(split_path),
    smiles_list=smiles_list,
    model_class=GCN,
    model_save_path=str(model_save_path),
    hidden_channels=hidden_channels,
    epochs=5,
    lr=0.001,
    batch_size=32,
    device=device_str,
)
print('Training result:', train_result)
print('Model saved exists:', model_save_path.exists())

Dataset CSV: C:\Repositories\AI-Agent-for-Compound-Prediction-and-Explainability\data\datasets\chembl_activity_data_O00329_P42336.csv
Split file : C:\Repositories\AI-Agent-for-Compound-Prediction-and-Explainability\splits\chembl_activity_data_O00329_P42336.pkl
SMILES count: 1195
Detected classes: 3
Using device: cuda
Training result: {'best_val_acc': 0.8511904761904762, 'test_acc': 0.841225626740947, 'model_path': 'C:\\Repositories\\AI-Agent-for-Compound-Prediction-and-Explainability\\session\\models\\gnn_edgeshaper_test.pt'}
Model saved exists: True


In [3]:
loaded_model = load_gnn_model(
    model_class=GCN,
    node_features_dim=4,
    hidden_channels=hidden_channels,
    num_classes=num_classes,
    model_path=str(model_save_path),
    device=device_str,
)

# Ensure model and graph tensors are on the same device
loaded_model = loaded_model.to(device)

sample_smiles = smiles_list[0]
sample_graph = nx_graph_to_pyg_data(smiles_to_nx_graph(sample_smiles), label=0)
assert sample_graph is not None, 'Could not build sample graph from SMILES'
sample_graph = sample_graph.to(device)

explainer = Edgeshaper(
    model=loaded_model,
    x=sample_graph.x,
    edge_index=sample_graph.edge_index,
    edge_weight=None,
    device=device_str,
)

with torch.no_grad():
    batch = torch.zeros(sample_graph.x.shape[0], dtype=torch.int64, device=device)
    pred_logits = loaded_model(sample_graph.x, sample_graph.edge_index, batch=batch)
    target_class = int(pred_logits.argmax(dim=1).item())

phi_edges = explainer.explain_batch(
    M=100,
    target_class=target_class,
    batch_size=100,
    seed=42,
    progress_bar=False,
)

orig_prob = explainer.compute_original_predicted_probability()
pp_set, infidelity = explainer.compute_pertinent_positive_set()
mk_set, fidelity = explainer.compute_minimal_top_k_set()
trust = explainer.compute_trustworthiness()

print('Target class:', target_class)
print('Num edges explained:', len(phi_edges))
print('Original predicted prob:', round(orig_prob, 6))
print('Fidelity+ :', round(fidelity, 6))
print('Infidelity:', round(infidelity, 6))
print('Trustworthiness:', round(trust, 6))
print('Pertinent positive edges:', pp_set.shape[1])
print('Minimal top-k edges:', mk_set.shape[1])

C:\Repositories\AI-Agent-for-Compound-Prediction-and-Explainability\src\chemagent\explainability\edgeshaper.py:474: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  E_z_mask = torch.tensor([rng.binomial(1, P, num_edges) for _ in range(batch_size)], dtype=torch.int32).flatten()


Target class: 0
Num edges explained: 80
Original predicted prob: 0.823206
Fidelity+ : 0
Infidelity: -0.025257
Trustworthiness: 0.0
Pertinent positive edges: 1
Minimal top-k edges: 80


In [4]:
# Optional visualization check (requires edgeshaper_viz_utils dependencies).
try:
    img_expl, _, _ = explainer.visualize_molecule_explanations(
        smiles=sample_smiles,
        save_path=None,
        pertinent_positive=False,
        minimal_top_k=False,
    )
    display(img_expl)
    print('Visualization path: OK')
except Exception as exc:
    print('Visualization skipped:', type(exc).__name__, '-', exc)

Visualization skipped: ImportError - EdgeSHAPer visualization helpers are unavailable. Check the edgeshaper_viz_utils dependencies to enable visualization.


In [5]:
# Minimal assertions for a quick regression smoke test
assert model_save_path.exists(), 'Model file was not saved'
assert isinstance(phi_edges, list) and len(phi_edges) == sample_graph.edge_index.shape[1]
assert isinstance(orig_prob, (int, float))
assert 0.0 <= float(orig_prob) <= 1.0
assert isinstance(fidelity, (int, float))
assert isinstance(infidelity, (int, float))
assert isinstance(trust, (int, float))
print('End-to-end GNN + EdgeSHAPer smoke test passed.')

End-to-end GNN + EdgeSHAPer smoke test passed.
